# 06 — RAG over the policy corpus

Builds a clause-level retriever over `policies/*.md` and tests it against questions
where the correct clause is known.

Two things this notebook establishes, both verified:

1. Splitting on `### ` gives exactly one clause per chunk — 23 chunks, 226–415 chars.
2. **A single blended "situation" query retrieves badly.** Asking one question that mixes
   charges + accounts + incorporation buries the clause that should obviously fire.
   Querying once per fact cluster fixes it. That finding is reproduced at the bottom.

In [3]:
%load_ext autoreload
%autoreload 2

import re, glob, os
import chromadb
from chromadb.utils import embedding_functions

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1 · Chunk — one clause per chunk

In [4]:
def load_clauses(policy_dir="policies"):
    """Split each policy .md into one chunk per clause, keyed on '### ' headings.

    Deliberately NOT a character-count splitter: those cut clauses in half and the
    model then reasons from a fragment. The documents were written so that heading
    boundaries are clause boundaries.
    """
    clauses = []
    for path in sorted(glob.glob(os.path.join(policy_dir, "*.md"))):
        raw = open(path, encoding="utf-8").read()
        doc = os.path.basename(path).split("_")[0]          # SEC / CON / EVD
        for block in re.split(r"\n(?=### )", raw):          # split before each '### '
            if not block.startswith("### "):
                continue                                     # skip the document preamble
            head = block.split("\n", 1)[0][4:].strip()      # 'SEC-01 — Outstanding...'
            m = re.search(r"\*\*Outcome:\s*([^*]+)\*\*", block)
            clauses.append({
                "clause_id": head.split(" ")[0],             # 'SEC-01'
                "doc":       doc,
                "title":     head,
                "outcome":   m.group(1).strip() if m else "",
                "text":      re.sub(r"\n---\s*$", "", block).strip(),
            })
    return clauses


clauses = load_clauses()
len(clauses)

23

In [7]:
# sanity check the chunking before embedding anything
ids = [c["clause_id"] for c in clauses]
assert len(set(ids)) == len(ids),              "duplicate clause ids"
assert all(c["text"].strip() for c in clauses), "empty chunk"
assert all(c["outcome"] for c in clauses),      "clause with no outcome"

print(f"{len(clauses)} clauses, all unique, all with an outcome")
print("chars  min %d / max %d" % (min(len(c["text"]) for c in clauses),
                                  max(len(c["text"]) for c in clauses)))
for c in clauses:
    print(f'  {c["clause_id"]:8} {c["outcome"][:24]:26} {c["title"][:50]}')

23 clauses, all unique, all with an outcome
chars  min 226 / max 462
  CON-01   REFER                      CON-01 — Late filing of annual accounts
  CON-02   DECLINE                    CON-02 — Materially late filing
  CON-03   REFER                      CON-03 — Pattern of late filing
  CON-04   REFER                      CON-04 — Micro-entity accounts
  CON-05   DECLINE                    CON-05 — No accounts on record
  CON-06   REFER                      CON-06 — Stale financial information
  CON-07   DECLINE                    CON-07 — Insolvency proceedings
  CON-08   PROCEED                    CON-08 — Paper filing
  EVD-01   INSUFFICIENT EVIDENCE —    EVD-01 — Minimum evidence set
  EVD-02   mandatory                  EVD-02 — Every assertion must be sourced
  EVD-03   mandatory                  EVD-03 — No external knowledge
  EVD-04   INSUFFICIENT EVIDENCE —    EVD-04 — Charge particulars before collateral stat
  EVD-05   PROCEED WITH QUALIFICATI   EVD-05 — Limit on research 

## 2 · Embed and store

Local SentenceTransformers model — no API cost, no network, and it's the family you
already benchmarked in the MSc NLP project. `all-MiniLM-L6-v2` is more than enough
for 23 chunks.

**`clause_id` goes in the metadata**, not just the text, so a retrieved chunk arrives
carrying its own citation and the agent can't misattribute a rule it read to a number
it half-remembers.

In [9]:
!pip install sentence_transformers

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------------------- 740.6/740.6 kB 7.7 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------- ----- 10.7/12.3 MB 61.0 MB/s eta 0:00:01
   ------------------------------------- -- 11.5/12.3 MB 31.4 MB/s eta 0:00:01
   ---------------------------------------  12.1/12.3 MB 21.6 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 17.5 MB/s  0:00:00
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   - -------------------------------------- 3.7/124.1 MB 19.8 MB/s eta 0:00:07
   -- ------------------------------------- 6.3/124.1 MB 14.9 MB/s eta 0:00:08
   --- ------------------------------------ 11.5/124.1 MB 18.5 MB/s eta 0:00:07
   ----- ---------------------------------- 16.8/124.1 MB 20.7 MB/s eta 0:00:06
   ------ -------

In [8]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.Client()                       # in-memory; fine at this size
coll = client.get_or_create_collection(
    "policy", embedding_function=ef, metadata={"hnsw:space": "cosine"}
)

coll.add(
    ids       =[c["clause_id"] for c in clauses],
    documents =[c["text"] for c in clauses],
    metadatas =[{"clause_id": c["clause_id"], "doc": c["doc"],
                 "title": c["title"], "outcome": c["outcome"]} for c in clauses],
)
coll.count()

ValueError: The sentence_transformers python package is not installed. Please install it with `pip install sentence_transformers`

In [ ]:
def retrieve(query, k=8, doc=None):
    """Nearest clauses to a query. `doc` restricts to SEC / CON / EVD."""
    res = coll.query(query_texts=[query], n_results=k,
                     where={"doc": doc} if doc else None)
    return [{"clause_id": m["clause_id"], "outcome": m["outcome"],
             "distance": round(d, 3), "text": t}
            for m, d, t in zip(res["metadatas"][0], res["distances"][0],
                               res["documents"][0])]


for r in retrieve("two outstanding charges to another lender", k=3):
    print(f'{r["clause_id"]:8} d={r["distance"]:.3f}  {r["outcome"]}')

## 3 · Test — recall@k against known answers

Eight questions where the correct clause is known. The metric is whether the right
clause appears in the top-k, which is checkable automatically — no reading required.
This is the Day 10 discipline, on a corpus small enough to eyeball.

In [ ]:
TESTS = [
 ("SEC-01", "Company has two outstanding charges held by Interbay Funding Limited, neither satisfied"),
 ("CON-01", "Accounts for the period ending 2024-06-30 were filed on 2025-12-17, months after the statutory deadline"),
 ("CON-04", "The company files micro-entity accounts which do not disclose turnover or profit"),
 ("SEC-02", "No charges have ever been registered against this company"),
 ("EVD-04", "The brief wants to state what the charge is secured against but particulars were not retrieved"),
 ("CON-05", "Company incorporated in 2017. No accounts filing appears anywhere in the record."),
 ("SEC-06", "Two charges created on the same day in favour of the same lender"),
 ("CON-07", "A liquidation filing appears in the company's history"),
]

ranks = []
for expect, q in TESTS:
    ids  = [r["clause_id"] for r in retrieve(q, k=8)]
    rank = ids.index(expect) + 1 if expect in ids else None
    ranks.append(rank)
    print(f'{"OK " if rank == 1 else "   "} expect {expect:8} rank {str(rank) if rank else ">8":4}  top3: {", ".join(ids[:3])}')

hit = lambda n: sum(1 for r in ranks if r and r <= n) / len(ranks)
print(f"\nrecall@1 {hit(1):.2f}   recall@3 {hit(3):.2f}   recall@5 {hit(5):.2f}   recall@8 {hit(8):.2f}")

Expected result: **recall@1 0.75, recall@3 0.88, recall@5 1.00.**

Two instructive misses:

- **`CON-01` loses to `CON-06`** by 0.01. "Filed late" and "period end is old" are
  different concepts sharing almost identical vocabulary. Essentially a coin flip.
- **`SEC-02` lands around rank 5.** It's about an *absence* of charges, and its nearest
  neighbours are other clauses about missing things (`EVD-01`, `CON-05`). The embedding
  matches on absence-language rather than on *what* is absent. `SEC-02` identifies your
  target population, so **`k=3` would lose the single most important clause in the corpus.**

Hence `k=8` as the floor. At 23 clauses that costs nothing.

## 4 · The blended-query problem

The policy node won't ask neat single-topic questions — it'll describe a whole company.
Watch what that does.

In [ ]:
situation = ("Company 10812571. Two charges outstanding, both created 2017-07-10 in favour "
             "of Interbay Funding Limited, neither satisfied. Micro-entity accounts, most "
             "recent period end 2024-06-30, filed 2025-12-17. Incorporated 2017-06-09.")

print("ONE BLENDED QUERY")
for r in retrieve(situation, k=6):
    print(f'  {r["clause_id"]:8} d={r["distance"]:.3f}  {r["outcome"]}')

In [ ]:
# One query per fact cluster, unioned. Same corpus, same model, different question shape.
FACETS = {
    "charges":  "two charges outstanding in favour of a third-party lender, neither satisfied",
    "accounts": "micro-entity accounts filed 8 months after the statutory deadline",
    "evidence": "is there enough evidence to make a recommendation",
}

found = {}
for facet, q in FACETS.items():
    for r in retrieve(q, k=3):
        found.setdefault(r["clause_id"], (r["outcome"], facet, r["distance"]))

print("FACETED")
for cid in sorted(found):
    outcome, facet, d = found[cid]
    print(f'  {cid:8} {outcome[:26]:28} via {facet:9} d={d:.3f}')
print(f"\n{len(found)} distinct clauses surfaced")

**The finding.** A long multi-fact query averages into a blurry vector — it sits between
clusters and matches nothing strongly, so `SEC-01` (two outstanding charges → REFER, the
clause that should obviously fire) drops to about rank 5 behind `CON-04`.

Split the same facts into one query per cluster and `SEC-01` comes back at the top of its
facet, at a *better* distance than anything the blended query produced.

So the `policy` node should compose **several short queries from the state**, not one
description of the company. Charges → one query. Accounts → another. Evidence
sufficiency → a third. Union the results and let the model decide which apply.

An honest alternative at this corpus size: skip retrieval for a facet and pass the whole
relevant document. 23 clauses is ~3,000 tokens. Retrieval earns its place at scale — the
reason to build it now is that the pattern is what a real policy corpus needs.

## 5 · Extract before Saturday

Once this runs clean, move `load_clauses`, the store construction and `retrieve` into
`policy_store.py` with two public functions:

```python
def build_store(policy_dir="policies") -> Collection
def retrieve(query, k=8, doc=None)     -> list[dict]
```

Then the `policy` node is an import rather than a copy-paste out of notebook cells.

**Build the store once**, at module level or node setup — never inside the node function.
Inside, you re-embed all 23 clauses for every company: invisible at n=1, painful at n=100
on Wednesday.